# Validation: predicting a known ternary complex (PDB 8G1P)

PDB **8G1P** is a crystal structure of the SMARCA2 bromodomain bound to the VHL complex
(VHL:Elongin-C:Elongin-B) by a PROTAC. This notebook takes only what one would have
*before* solving such a structure - the two proteins and the PROTAC as SMILES - predicts
the ternary complex, and scores the predicted poses against the crystal with
[DockQ](https://github.com/bjornwallner/DockQ).

CAPRI quality classes: DockQ >= 0.23 acceptable, >= 0.49 medium, >= 0.80 high.

**Read the result before reusing this for your own system.** On this case the predicted
poses do not reproduce the crystal arrangement (see the table at the end). On 68 PROTAC
ternary crystal structures, the top-ranked pose was CAPRI-acceptable in 2 of the 31 that
could be docked from SMILES. Treat the output as candidates to be filtered by other evidence
(known binding epitopes, mutagenesis, MD stability), not as a prediction to trust as is.

Runtime: about 10 minutes.

## Setup
Checks that PolyLego, GROMACS and AmberTools are available and pins the thread counts.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(os.path.join("..", "common")))
from polylego_examples import *

# Pin threads for every step launched from this notebook (see examples/README.md).
os.environ.setdefault("OMP_NUM_THREADS", "8")
for k in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[k] = "1"
os.environ.setdefault("PYTHONHASHSEED", "0")

GMX = check_env()
os.environ["POLYLEGO_GMX"] = GMX
# GPU for the GROMACS runs: "auto" (use a compatible GPU if there is one, else CPU),
# "none" (CPU only), or a GPU id such as "0".
GPU = os.environ.get("POLYLEGO_GPU", "auto")
os.makedirs("output", exist_ok=True)

PolyLego   1.4.0  ($CONDA_PREFIX/lib/python3.10/site-packages/PolyLego)
Python     3.10.13
GROMACS    $CONDA_PREFIX/bin/gmx
AmberTools $CONDA_PREFIX/bin/antechamber
OMP_NUM_THREADS        8
OPENBLAS_NUM_THREADS   1
MKL_NUM_THREADS        1


## 1. Inputs
The crystal chains (SMARCA2 bromodomain; VHL:EloC:EloB) and the PROTAC as SMILES. The crystal PROTAC is loaded only as a reference for the figures and the scoring - it is not used as an input to the prediction.

In [2]:
smiles = open("input/protac.smi").read().split()[0]
print(smiles)
print(open("input/protac.smi").read().split()[1], "- ligand code in the PDB entry")

Cc1c(scn1)c2ccc(c(c2)OCCc3ccc(cc3)CN4CCN(CC4)c5cc(nnc5N)c6ccccc6O)CNC(=O)[C@@H]7C[C@H](CN7C(=O)[C@H](C(C)(C)C)NC(=O)C8(CC8)F)O
FWZ - ligand code in the PDB entry


In [3]:
show_complex("input/e3.pdb", ligands=["input/protac_crystal.pdb"],
             focus_ligand=False)   # VHL complex + the crystal PROTAC

[3D view - run the notebook in Jupyter to display it]


## 2. PROTAC conformer from SMILES
A 3D conformer generated with RDKit (ETKDG + MMFF). No crystal coordinates enter here.

In [4]:
sdf = smiles_to_sdf(smiles, "output/protac.sdf", name="FWZ")
show_mol(sdf)

[3D view - run the notebook in Jupyter to display it]


## 3. Binding sites
P2Rank pockets shipped in `input/pockets/`. How far are they from where the PROTAC actually binds in the crystal?

In [5]:
import numpy as np
E3_SITE = pocket_center("input/pockets/e3.pdb_predictions.csv")
POI_SITE = pocket_center("input/pockets/poi.pdb_predictions.csv")
xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                for l in open("input/protac_crystal.pdb")
                if l.startswith(("ATOM", "HETATM"))])
for tag, site in (("VHL", E3_SITE), ("SMARCA2", POI_SITE)):
    d = np.linalg.norm(xyz - np.array(site), axis=1).min()
    print(f"{tag:8s} pocket {np.round(site, 1)}  {d:5.1f} A to the nearest crystal PROTAC atom")

VHL      pocket [-27.6 -13.4  30.3]    1.5 A to the nearest crystal PROTAC atom
SMARCA2  pocket [-38.1 -15.7  31.1]    0.5 A to the nearest crystal PROTAC atom


## 4. Predict the ternary complex
The two warhead ends are docked into their pockets, linker conformers are sampled, and the resulting protein arrangements are scored and refined. `--topn 5` keeps the five best-scoring poses.

In [6]:
run(["PolyLego-OptimizeBinding", "--protacs", "--ensemble",
     "--input_path", "input/", "--output_path", "output/",
     "--protein1", "e3.pdb", "--protein2", "poi.pdb",
     "--protac_mol", sdf, "--docking_site1", *E3_SITE, "--docking_site2", *POI_SITE,
     "--name", "8G1P", "--topn", 5], log="output/dock.log", tail=10)

$ PolyLego-OptimizeBinding --protacs --ensemble --input_path input/ --output_path output/ --protein1 e3.pdb --protein2 poi.pdb --protac_mol output/protac.sdf --docking_site1 -27.6091 -13.4179 30.3287 --docking_site2 -38.1159 -15.7389 31.0849 --name 8G1P --topn 5


  5 poses (warhead / E3-ligand split: SMARTS)
    rank         LJ  contacts  clashes  shift(A)
       1      -23.2       141        0      0.01
       2      -21.1       149        0      0.02
       3      -15.3       118        0      0.01
       4      -14.9       114        0      0.01
       5      -15.4       102        0      0.02
  Candidates, not a prediction: on 68 PROTAC ternary crystal structures the top-ranked pose was CAPRI-acceptable in 2 of the 31 docked from SMILES.
  Finished in 1 min 57 s
    output/docking_ensemble
  -> exit 0, 119 s


## 5. Score against the crystal

In [7]:
import glob, pandas as pd
reference = write_reference_complex("input/poi.pdb", "input/e3.pdb", "output/reference.pdb")
rows = []
for m in sorted(glob.glob("output/docking_ensemble/*.pdb")):
    r = dockq(m, reference, {"G"}, {"A", "B", "C", "E"})
    rows.append({"pose": m.split("/")[-1],
                 "DockQ": round(r["DockQ"], 3) if r else 0.0,
                 "fnat": round(r.get("fnat", 0.0), 3) if r else 0.0,
                 "iRMSD (A)": round(r.get("iRMSD", float("nan")), 2) if r else None})
df = pd.DataFrame(rows)
df["CAPRI"] = [capri_label(v) for v in df["DockQ"]]
df

,pose,DockQ,fnat,iRMSD (A),CAPRI
0,8G1P_001.pdb,0.034,0.000,13.99,incorrect
1,8G1P_002.pdb,0.051,0.045,8.33,incorrect
2,8G1P_003.pdb,0.041,0.000,7.00,incorrect
3,8G1P_004.pdb,0.027,0.000,9.47,incorrect
4,8G1P_005.pdb,0.049,0.000,9.52,incorrect


Top-ranked pose (magenta) against the crystal arrangement. The VHL complex is the common frame; the difference you see is where SMARCA2 ends up.

In [8]:
show_complex("output/reference.pdb", ligands=["output/docking_ensemble/8G1P_001.pdb"],
             focus_ligand=False)

[3D view - run the notebook in Jupyter to display it]


## 6. What this shows

The protocol runs end to end and is reproducible from public inputs, and the
poses are physically reasonable as structures. On this case they are **not** the
crystal arrangement: the scoring function does not single out the native-like pose.

Use the candidate list with independent evidence, and re-run this notebook on a
system of your own to see what the method does with inputs you can check.